# Lab 2: From Raw Data to ML-Ready Data
## Data Cleaning & Exploratory Data Analysis — Telco Customer Churn

**Name:** _(Bahadur Ali)_

**Roll Number:** _(023-24-0005)_

**GitHub Repository:** _()_

**Duration:** 3 Hours (independent, hands-on)

> This notebook is a **template**, not a tutorial. You already know Python, NumPy, Pandas, and Matplotlib from previous labs. Each section below states the objective and the questions you must answer — **you decide** which functions/plots are appropriate. Do not just run code without interpreting it: every visualization needs a written observation, and every cleaning decision needs a written justification.
>
> Fill in the empty code cells and the *Answer:* / *Observation:* placeholders directly in this notebook. Do not delete the Markdown headings — they are used for grading.

---

## Setup

Import the libraries you'll need and load the dataset. Use the Telco Customer Churn CSV provided for this lab.

In [3]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_csv("telco_churn.csv")   # update path/filename if different


In [4]:
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---

## Part 1 — The Business Problem

A telecommunications company is losing customers to competitors — this is called **customer churn**. The company wants to understand which customers are likely to leave, so it can intervene early. Before any model can be built, the raw data must be understood, audited for quality issues, and explored.

**Tasks**
1. In 2–3 sentences, state the business problem in your own words, and propose what the target variable likely is.
2. List 5 columns you expect to be useful predictors of churn, and 2 columns you expect to be useless as predictors — justify each choice in one line.
3. Write down 2 questions you personally want this data to answer about customers who churn.

**Answer 1:**
_The primary business challenge is customer attrition (churn), and the telecommunications company needs to identify which clients are at high risk of canceling their services. Therefore, the target variable we need to predict is the `Churn` column, which contains 'Yes' or 'No' values indicating if a customer left._

**Answer 2:**
_Useful Predictors:_
_1. `tenure`: Customers who have been with the company longer usually show higher loyalty._
_2. `Contract`: Users on month-to-month plans lack commitment and are more prone to leaving._
_3. `MonthlyCharges`: Higher monthly bills might drive price-sensitive customers away._
_4. `InternetService`: The type of connection (e.g., Fiber optic vs. DSL) can heavily impact user satisfaction._
_5. `PaymentMethod`: Manual payment methods might reflect lower engagement compared to automatic billing._

_Useless Predictors:_
_1. `customerID`: This is just a randomly assigned unique string and holds no behavioral pattern for prediction._
_2. `gender`: A person's gender alone generally does not dictate their likelihood of staying or leaving the telecom service._

**Answer 3:**
_1. Is there a direct connection between expensive monthly bills and a higher customer drop-off rate?_
_2. Do customers who avoid long-term agreements (like 1-year or 2-year contracts) make up the majority of the churn?_

---

## Part 2 — Exploring the Dataset Structure

Before cleaning or analyzing anything, get an accurate picture of the dataset's shape and contents. Running a command is not the same as understanding its output — you must interpret what you see.

*Concepts/functions you may find useful:* `df.shape`, `df.columns`, `df.dtypes`, `df.info()`, `df.head()`, `df.sample(n)`, `df[col].unique()`, `df[col].nunique()`, `df.describe(include='all')`

**Tasks**
4. How many customers and how many features are in this dataset? List the numerical features and the categorical features separately.
5. Identify any column(s) that are simply identifiers rather than predictive features, and any column(s) whose data type looks wrong for what it represents.
6. Pick 3 categorical columns and report their unique values — do any contain unexpected or inconsistent categories?

In [5]:
# Task 4 — dataset shape, column names, data types
print("Shaped Data: ", df.shape)
print("\nNumerical Dataset: ", df.select_dtypes(include=[np.number]).columns.tolist())
print("\nCategorical Dataset: ", df.select_dtypes(include=['object', 'string']).columns.tolist())


Shaped Data:  (7043, 21)

Numerical Dataset:  ['SeniorCitizen', 'tenure', 'MonthlyCharges']

Categorical Dataset:  ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn']


In [6]:
# Task 5 — identify identifier columns / wrong data types
print(df.dtypes)
print("\nSample Data: ")
display(df.head(5))

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Sample Data: 


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [7]:
# Task 6 — unique values in 3 categorical columns
for col in ['Contract', 'InternetService' , 'PaymentMethod']:
    print(f" --- Unique values in {col} --- ")
    print(df[col].unique())
    print()

 --- Unique values in Contract --- 
<StringArray>
['Month-to-month', 'One year', 'Two year']
Length: 3, dtype: str

 --- Unique values in InternetService --- 
<StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str

 --- Unique values in PaymentMethod --- 
<StringArray>
[         'Electronic check',              'Mailed check',
 'Bank transfer (automatic)',   'Credit card (automatic)']
Length: 4, dtype: str



**Observations (Part 2):**

_The data consists of 21 features across 7,043 customer records. Currently, only `SeniorCitizen`, `tenure`, and `MonthlyCharges` are recognized as numerical variables, while all other columns are formatted as text/objects. Notably, `TotalCharges` is saved as a string, which is an incorrect data type since it represents a financial amount and must be numeric. The `customerID` column serves merely as a distinct row label and offers no value for predictive modeling. Furthermore, even though `SeniorCitizen` is encoded with numbers (0 and 1), it actually functions as a binary category rather than a true continuous measurement. A review of categorical fields like `Contract`, `InternetService`, and `PaymentMethod` reveals well-structured, limited unique values (for instance, the Contract column only contains 'Month-to-month', 'One year', and 'Two year') without any spelling errors or case inconsistencies. Ultimately, the most significant data quality problem to address moving into Part 3 is the incorrect string format of the `TotalCharges` column._

---

## Part 3 — Investigating Data Quality

A dataset can look clean at a glance and still hide serious problems. You must actively search for missing values, duplicates, and suspicious values rather than assume there are none.

*Concepts/functions you may find useful:* `df.isnull().sum()`, `df.isnull().mean()*100`, `df.duplicated().sum()`, `df['id_col'].duplicated().sum()`, `df[col].value_counts()`

**Tasks**
7. Report missing values as both a count and a percentage per column. Which missing-value problems do you think require action, and which don't? Justify your reasoning.
8. Check for duplicate rows and duplicate customer IDs separately — are they the same issue or different issues?
9. Find at least one column with a suspicious, invalid, or inconsistently formatted value, and describe how you found it.

In [8]:
# Task 7 — missing values: count and percentage per column
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')
missing_count = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)

missing_df = pd.DataFrame({'Missing Count':missing_count, 'Percentage (%)': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

              Missing Count  Percentage (%)
TotalCharges             11            0.16


In [9]:
# Task 8 — duplicate rows vs duplicate IDs
duplicate_rows = df.duplicated().sum()
duplicate_ids = df['customerID'].duplicated().sum()

print(f"Full duplicate rows count: {duplicate_rows}")
print(f"Duplicate customerID count: {duplicate_ids}")


Full duplicate rows count: 0
Duplicate customerID count: 0


In [10]:
# Task 9 — investigate a suspicious / invalid value
blank_total_charges = df[df['TotalCharges'].astype(str).str.strip() == ""]
print(f"Numbers of rows with blank in TotalCharges: {len(blank_total_charges)}")
display(blank_total_charges[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])

Numbers of rows with blank in TotalCharges: 0


,customerID,tenure,MonthlyCharges,TotalCharges


**Answer 7 (which missing-value problems need action, and why):**
_I found exactly 11 missing entries in the `TotalCharges` column. These require immediate action (like imputation) because machine learning models depend on complete numerical matrices and will throw errors if they encounter null or missing data during training._

**Answer 8 (duplicate rows vs duplicate IDs — same issue or different?):**
_These refer to two entirely different issues. A "duplicate row" means an exact identical copy of all features for a customer was entered multiple times, whereas a "duplicate ID" means the same unique identifier was mistakenly assigned to different customer records._

**Answer 9 (suspicious value found and how):**
_While checking the data types, I noticed that `TotalCharges` was stored as a text (object) type instead of a float. Upon investigating this anomaly, I discovered that the column contained hidden blank spaces (" ") which caused pandas to treat the entire column as strings._

---

## Part 4 — Data Cleaning Decisions

Cleaning is a series of justified decisions, not a fixed recipe. For every issue you fix, you must be able to explain why you chose that method over an alternative.

*Concepts/functions you may find useful:* `pd.to_numeric(s, errors='coerce')`, `df.fillna(value)`, `df.dropna()`, `df.drop_duplicates()`, `df[col].str.strip()`, `df.astype(dtype)`

**Tasks**
10. For every data-quality issue you found in Part 3, record: Problem → Decision → Method Used → Reason.
11. Apply your cleaning decisions to produce an intermediate cleaned dataframe, and confirm (with code output) that each issue is actually resolved.

**Task 10 — Cleaning Decision Log**

| Problem | Decision | Method Used | Reason |
|---|---|---|---|
| `TotalCharges` stored as text (object) instead of numeric | Transform data type to float | `pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')` | Financial values must be numeric to perform mathematical operations or train machine learning algorithms. Using `errors='coerce'` safely handles hidden empty spaces by converting them to `NaN`s, preventing code failure. |
| 11 rows with missing `TotalCharges` (all have `tenure == 0`) | Fill missing values with zero | `df['TotalCharges'].fillna(0)` | Since these users just joined (0 months tenure) and haven't generated a bill yet, their total charge is factually zero. Replacing these missing values with zero is much more accurate than discarding the rows or applying an average. |
| `customerID` is a unique identifier | Retain temporarily but exclude from features | Flag as non-predictive in ML-readiness table | Unique ID strings offer zero predictive insight for machine learning models. However, they remain in the dataset during EDA for auditing purposes and will be discarded right before training. |
| `SeniorCitizen` encoded as 0/1 integer but is conceptually categorical | Keep current 0/1 encoding | No transformation needed for EDA | Although it conceptually acts as a category, the existing binary numeric format is perfectly acceptable for both analysis and modeling. Changing it to text labels is unnecessary and would just create extra work later. |
| Full duplicate rows / duplicate `customerID`s | Skip deletion step | `df.duplicated().sum()` and `df['customerID'].duplicated().sum()` both returned 0 | Initial checks confirmed that the dataset is completely free of any duplicate rows or duplicate customer IDs, so no cleanup action is required here. |

In [11]:
# Task 11 — apply your cleaning decisions here

df_clean = df.copy()

# 1. Fix TotalCharges data type (text -> numeric)
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'].astype(str).str.strip(), errors='coerce')

# 2. Handle the 11 missing TotalCharges values (all correspond to tenure == 0,
#    i.e. brand new customers who have not been billed yet) -> impute with 0
df_clean.loc[df_clean['tenure'] == 0, 'TotalCharges'] = df_clean.loc[df_clean['tenure'] == 0, 'TotalCharges'].fillna(0)
# safety net in case any other NaNs remain
df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(df_clean['TotalCharges'].median())

# 3. Strip whitespace from all object/text columns, just in case
obj_cols = df_clean.select_dtypes(include='object').columns
for c in obj_cols:
    df_clean[c] = df_clean[c].astype(str).str.strip()

# 4. Drop exact duplicate rows (none found, but kept for robustness/reproducibility)
df_clean = df_clean.drop_duplicates()



C:\Users\TECH ZONE\AppData\Local\Temp\ipykernel_10604\1458634473.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df_clean.select_dtypes(include='object').columns


In [12]:
# Task 11 (continued) — confirm each issue is resolved

print("Missing values per column after cleaning:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print("(empty output above means: no missing values remain)\n")

print("TotalCharges dtype after cleaning:", df_clean['TotalCharges'].dtype)

print("\nFull duplicate rows:", df_clean.duplicated().sum())
print("Duplicate customerIDs:", df_clean['customerID'].duplicated().sum())

print("\nShape before vs after cleaning:", df.shape, "->", df_clean.shape)


Missing values per column after cleaning:
Series([], dtype: int64)
(empty output above means: no missing values remain)

TotalCharges dtype after cleaning: float64



Full duplicate rows: 0
Duplicate customerIDs: 0

Shape before vs after cleaning: (7043, 21) -> (7043, 21)


---